# Practica 3 - Visualizacion de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

path = "../data/procesed/metro.csv" 
metro = pd.read_csv(path)
metro.head()

## Grafica de barras

In [ ]:
def bar_char(x, y, title: str = "bar char", x_label: str = "", y_label: str = "", palette: str = "viridis", figsize: tuple = (12, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    n_bars = len(x)
    barras = ax.bar(x, y, color=sns.color_palette(palette, n_bars))
    ax.bar_label(barras, padding=3)
    ax.set_title(title)
    ax.set_xlabel(x_label if x_label else (x.name if hasattr(x, "name") and x.name else ""))
    ax.set_ylabel(y_label if y_label else (y.name if hasattr(y, "name") and y.name else ""))
    plt.xticks(rotation=45 if n_bars > 5 else 0)
    fig.tight_layout()
    return fig, ax

In [ ]:
weather_counts = metro["weather_main"].value_counts()
bar_char(
    x=weather_counts.index,
    y=weather_counts.values,
    title="frecuencia por tipo de clima principal (weather_main)",
    x_label="Tipo de clima",
    y_label="cantidad de registros",
    palette="viridis"
)

## Grafica de dispersion

In [ ]:
def scatter_plot(x, y, c=None, title: str = "scatter plot", x_label: str = "", y_label: str = "", c_label: str = "", cmap: str = "coolwarm", alpha: float = 0.7, figsize: tuple = (8, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    color_val = c if c is not None else y
    sc = ax.scatter(x, y, c=color_val, cmap=cmap, alpha=alpha, edgecolor="k")
    
    colorbar_label = c_label if c_label else (c.name if hasattr(c, "name") and c.name else (y_label if y_label else (y.name if hasattr(y, "name") and y.name else "Valor")))
    fig.colorbar(sc, ax=ax, label=colorbar_label)
    
    xlabel_text = x_label if x_label else (x.name if hasattr(x, "name") and x.name else "x")
    ylabel_text = y_label if y_label else (y.name if hasattr(y, "name") and y.name else "y")
    
    ax.set(title=title, xlabel=xlabel_text, ylabel=ylabel_text)
    fig.tight_layout()
    return fig, ax

In [ ]:
scatter_plot(
    x=metro["temp"],
    y=metro["traffic_volume"],
    title="Relacion entre temperatura y volumen de trafico",
    x_label="temperatura (K)",
    y_label="volumen de trafico",
    c_label="Volumen de trafico",
    cmap="coolwarm"
)

## Grafica de caja y violin

In [ ]:
def grafica_box_violin(data: pd.DataFrame, x: str, y: str, title: str = "Distribucion por grupo (Boxplot y Violin)", x_label: str = "", y_label: str = "", palette: str = "Set2", figsize: tuple = (12, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)
    sns.boxplot(data=data, x=x, y=y, hue=x, ax=axes[0], palette=palette, legend=False)
    axes[0].set_title("boxplot")
    axes[0].set_xlabel(x_label if x_label else x)
    axes[0].set_ylabel(y_label if y_label else y)
    axes[0].tick_params(axis="x", rotation=45)

    sns.violinplot(data=data, x=x, y=y, hue=x, ax=axes[1], palette=palette, legend=False)
    axes[1].set_title("violin plot")
    axes[1].set_xlabel(x_label if x_label else x)
    axes[1].set_ylabel("")
    axes[1].tick_params(axis="x", rotation=45)

    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes

In [ ]:
top_weather = metro[metro["weather_main"].isin(["Clouds", "Clear", "Rain", "Mist", "Snow"])]
grafica_box_violin(
    data=top_weather,
    x="weather_main",
    y="traffic_volume",
    title="distribucion del volumen de trafico segun el clima",
    x_label="tipo de clima",
    y_label="Volumen de trafico",
    palette="Set2"
)

## Mapa de calor

In [ ]:
def grafica_heatmap(data: pd.DataFrame, columns: list = None, title: str = "Mapa de calor", cmap: str = "vlag", figsize: tuple = (8, 6), annot: bool = True):
    if columns is not None:
        df_subset = data[columns]
    else:
        df_subset = data.select_dtypes(include=[np.number])
    
    corr = df_subset.corr()
    mascara = np.triu(np.ones_like(corr, dtype=bool))
    
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(corr, mask=mascara, annot=annot, fmt=".2f", cmap=cmap, vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    return fig, ax

In [ ]:
grafica_heatmap(
    data=metro,
    columns=["temp", "rain_1h", "snow_1h", "clouds_all", "traffic_volume"],
    title="matriz de Correlacion - Variables del Trafico Metro",
    cmap="coolwarm"
)

## Grafica de lineas

In [ ]:
def grafica_linea(x, y, title: str = "Grafica de lineas", x_label: str = "", y_label: str = "", label = None, linestyle: str = "-", linewidth: float = 2, figsize: tuple = (10, 4)):
    fig, ax = plt.subplots(figsize=figsize)
    if isinstance(y, dict):
        for name, val in y.items():
            ax.plot(x, val, label=name, linewidth=linewidth)
        ax.legend()
    elif isinstance(y, (list, tuple)) and len(y) > 0 and isinstance(y[0], (pd.Series, np.ndarray, list)):
        for idx, val in enumerate(y):
            lbl = label[idx] if isinstance(label, list) and idx < len(label) else f"Serie {idx+1}"
            ax.plot(x, val, label=lbl, linewidth=linewidth)
        ax.legend()
    else:
        lbl = label if label else (y.name if hasattr(y, "name") and y.name else "valor")
        ax.plot(x, y, label=lbl, linestyle=linestyle, linewidth=linewidth)
        if label or hasattr(y, "name"):
            ax.legend()

    xlabel_text = x_label if x_label else (x.name if hasattr(x, "name") and x.name else "x")
    ylabel_text = y_label if y_label else (y.name if hasattr(y, "name") and y.name else "y")

    ax.set(title=title, xlabel=xlabel_text, ylabel=ylabel_text)
    fig.tight_layout()
    return fig, ax

In [ ]:
metro_sample = metro.iloc[:100].copy()
metro_sample["date_time"] = pd.to_datetime(metro_sample["date_time"])

grafica_linea(
    x=metro_sample["date_time"],
    y=metro_sample["traffic_volume"],
    title="Evolucion del volumen de trafico (primeras 100 horas)",
    x_label="Fecha y hora",
    y_label="volumen de trafico",
    label="Volumen de trafico",
    linewidth=2
)